# Step 9 — RAG Service Prototype

This notebook combines the entire pipeline into an `answer_question()` function.

Flow:

1. Accept a question.
2. Retrieve Top-K chunks.
3. Assemble context.
4. Build prompt.
5. Send to LLM (optional).
6. Return answer with sources.

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
from pathlib import Path
import sqlite3
import numpy as np
from sentence_transformers import SentenceTransformer

DB_PATH = Path("../data/linux_docs.db")

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

MODEL_NAME = "BAAI/bge-small-en-v1.5"
embedder = SentenceTransformer(MODEL_NAME)
TOP_K = 5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2894.75it/s]


In [3]:
def retrieve(question: str, top_k: int = TOP_K):
    qvec = embedder.encode(question, normalize_embeddings=True).astype(np.float32)

    cursor.execute("""
    SELECT
        p.url,
        p.title,
        c.section,
        c.content,
        e.embedding
    FROM embeddings e
    JOIN chunks c ON e.chunk_id = c.id
    JOIN pages p ON c.page_id = p.id
    """)

    results = []

    for row in cursor.fetchall():
        vec = np.frombuffer(row["embedding"], dtype=np.float32)
        score = float(np.dot(qvec, vec))

        results.append({
            "score": score,
            "url": row["url"],
            "title": row["title"],
            "section": row["section"],
            "content": row["content"]
        })

    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_k]


In [4]:
def build_prompt(question: str, contexts: list[dict]) -> str:
    context = "\n\n".join(
        f"[Source: {c['title']} | {c['section']}]\n{c['content']}"
        for c in contexts
    )

    return f"""You are a Linux documentation assistant.

Use ONLY the context below to answer.

If the answer cannot be found, say you don't know.

Context:
{context}

Question:
{question}

Answer:
"""


In [5]:
def answer_question(question: str):
    contexts = retrieve(question)

    prompt = build_prompt(question, contexts)

    return {
        "question": question,
        "prompt": prompt,
        "sources": [
            {
                "title": c["title"],
                "section": c["section"],
                "url": c["url"],
                "score": round(c["score"], 4)
            }
            for c in contexts
        ]
    }


In [6]:
question = "How do I partition the disk during Arch Linux installation?"

result = answer_question(question)

print("QUESTION")
print(result["question"])

print("\nPROMPT PREVIEW")
print(result["prompt"][:2000])

print("\nSOURCES")
for src in result["sources"]:
    print(src)


QUESTION
How do I partition the disk during Arch Linux installation?

PROMPT PREVIEW
You are a Linux documentation assistant.

Use ONLY the context below to answer.

If the answer cannot be found, say you don't know.

Context:
[Source: Installation guide - ArchWiki | Introduction]
This document is a guide for installing [Arch Linux](/title/Arch_Linux) using the live system booted from an installation medium made from an official installation image. Its accessibility features are described on the page [Install Arch Linux with accessibility options](/title/Install_Arch_Linux_with_accessibility_options). For alternative means of installation, see [Category:Installation process](/title/Category:Installation_process).

Before installing, it would be advised to view the [FAQ](/title/FAQ). For conventions used in this document, see [Help:Reading](/title/Help:Reading). In particular, code examples may contain placeholders (formatted in *italics*

This guide is kept concise and you are advised 

## Send prompt to LLM (Optional)

If you want to generate answers automatically, send `result["prompt"]` to a model of your choice.

Example with OpenAI SDK:

```python
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-4.1",
    input=result["prompt"]
)

print(response.output_text)
```

Or use Ollama, LM Studio, or other providers by replacing the model call section.

In [7]:
conn.close()
print("Done ✅")

Done ✅
